In [1]:
import pandas as pd, numpy as np
from pathlib import Path
import os
# ROOT dinámico — funciona en local, CI y cualquier entorno
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

SILVER = Path("output/ambiental/02-silver")
GOLD   = Path("output/ambiental/03-gold")
GOLD.mkdir(parents=True, exist_ok=True)
print("✅ Rutas OK")

✅ Rutas OK


In [2]:
df = pd.read_parquet(SILVER / "clean_aemet_provincia_fecha.parquet")

# La col 'anio' parece tener solo 2026 (API live) — verificar con fecha real
df["fecha"] = pd.to_datetime(df["fecha"])
df["anyo_real"] = df["fecha"].dt.year

print(f"Shape: {df.shape}")
print(f"Años reales en fecha: {sorted(df['anyo_real'].unique())}")
print(df.head(3).to_string())

Shape: (41, 12)
Años reales en fecha: [np.int32(2026)]
  cod_provincia provincia      fecha  n_estaciones  ta_media  tamin_media  tamax_media  prec_total  hr_media  ta_std  anio  anyo_real
0            01     Álava 2026-04-14            88      9.39         8.34         9.67         0.0     69.31    4.63  2026       2026
1            02  Albacete 2026-04-14            75     12.41        11.68        12.64         0.0     72.04    4.64  2026       2026
2            03  Alicante 2026-04-14           115      9.39         8.25         9.61         0.0     65.79    6.58  2026       2026


In [3]:
df_gold = (
    df.groupby(["provincia", "anyo_real"], as_index=False)
    .agg(
        ta_media_anual   = ("ta_media",    "mean"),
        tamin_media_anual= ("tamin_media", "mean"),
        tamax_media_anual= ("tamax_media", "mean"),
        prec_total_anual = ("prec_total",  "sum"),
        hr_media_anual   = ("hr_media",    "mean"),
        n_obs            = ("n_estaciones","sum"),
    )
    .rename(columns={"anyo_real": "anyo"})
)

# Redondear
for c in ["ta_media_anual", "tamin_media_anual", "tamax_media_anual",
          "prec_total_anual", "hr_media_anual"]:
    df_gold[c] = df_gold[c].round(2)

print(f"Shape: {df_gold.shape}")
print(f"Años: {sorted(df_gold['anyo'].unique())}")
print(df_gold.head(5).to_string())

Shape: (41, 8)
Años: [np.int32(2026)]
   provincia  anyo  ta_media_anual  tamin_media_anual  tamax_media_anual  prec_total_anual  hr_media_anual  n_obs
0   A Coruña  2026           14.25              13.77              14.43               4.0           70.94    982
1   Albacete  2026           12.41              11.68              12.64               0.0           72.04     75
2   Alicante  2026            9.39               8.25               9.61               0.0           65.79    115
3    Almería  2026           14.05              13.55              14.16               0.0           47.34     60
4  Andalucía  2026            8.17               7.31               8.37               3.3           83.11    137


In [4]:
df_gold.to_parquet(GOLD / "gold_ambiental_provincia_anio.parquet", index=False)
df_gold.to_csv(GOLD    / "gold_ambiental_provincia_anio.csv",     index=False)

print(f"✅ Guardado en {GOLD}")
print(f"   gold_ambiental_provincia_anio → {df_gold.shape}")
print(f"\nDtypes:\n{df_gold.dtypes}")

✅ Guardado en output/ambiental/03-gold
   gold_ambiental_provincia_anio → (41, 8)

Dtypes:
provincia                str
anyo                   int32
ta_media_anual       float64
tamin_media_anual    float64
tamax_media_anual    float64
prec_total_anual     float64
hr_media_anual       float64
n_obs                  int64
dtype: object
